# CARE fault forecast: NVIDIA Kumo Tabular on the CatBoost 15-minute shortlist (full)

Same anchors, censored target (onset within 7 days), folds and inner groups as the CatBoost
15-minute model of the Smarthack repository. Kumo is in-context: each fit is a sample of the
training turbines' anchors as context (at most 100 columns, 10,240 rows per estimator, inside
the model's training range), the query is the anchors to score. Scores are evaluated locally
with the baseline protocol.

In [ ]:
import os, sys, subprocess, torch
print(sys.version.split()[0], "| torch", torch.__version__, "| CPUs", os.cpu_count())
assert torch.cuda.is_available(), "no GPU"
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
!pip install -q structured-data-models 2>&1 | tail -3
import sdm; print('sdm', getattr(sdm, '__version__', '?'))

In [ ]:
import base64, os
os.makedirs(os.path.dirname('/kaggle/working/fault_detection/care_fault_forecast.py'), exist_ok=True)
open('/kaggle/working/fault_detection/care_fault_forecast.py', 'wb').write(base64.b64decode('...'))

In [ ]:
import base64, os
os.makedirs(os.path.dirname('/kaggle/working/fault_detection/care_catboost_15min.py'), exist_ok=True)
open('/kaggle/working/fault_detection/care_catboost_15min.py', 'wb').write(base64.b64decode('...'))

In [ ]:
import base64, os
os.makedirs(os.path.dirname('/kaggle/working/fault_detection/care_kumo_tabular.py'), exist_ok=True)
open('/kaggle/working/fault_detection/care_kumo_tabular.py', 'wb').write(base64.b64decode('IiIiCiMgQ0FSRSBmYXVsdCBmb3JlY2FzdDogTlZJRElBIEt1bW8gVGFidWxhciAoaW4tY29udGV4dCkgb24gdGhlIENhdEJvb3N0IDE1LW1pbnV0ZSBzaG9ydGxpc3QKClJ1bnMgb24gYSBDVURBIEdQVSAoS2FnZ2xlKSwgYWZ0ZXIgYGNhcmVfZmF1bHRfZm9yZWNhc3QucHkgcHJlcGFyZWAgYW5kCmBjYXJlX2NhdGJvb3N0XzE1bWluLnB5IGZlYXR1cmVzYCBoYXZlIGJ1aWx0IHRoZSBhbmNob3JzIGFuZCBmZWF0dXJlcyB0aGVyZS4KClNhbWUgYW5jaG9ycywgdGFyZ2V0IChvbnNldCB3aXRoaW4gNyBkYXlzLCBjZW5zb3JlZCksIGZvbGRzIChgY3ZfZm9sZHMucGFycXVldGAsIGBwZXJfZmFybWApCmFuZCBpbm5lciBncm91cHMgYXMgYGNhcmVfY2F0Ym9vc3RfMTVtaW4ucHkgY3ZgLiBLdW1vIGlzIG5vdCB0cmFpbmVkOiBmb3IgZXZlcnkgZml0IHRoZQoiY29udGV4dCIgaXMgYSBzYW1wbGUgb2YgdGhlIHRyYWluaW5nIHR1cmJpbmVzJyBsYWJlbGxlZCBhbmNob3JzIGFuZCB0aGUgInF1ZXJ5IiBpcyB0aGUgYW5jaG9ycwp0byBzY29yZS4gUGVyIG91dGVyIGZvbGQ6IDMgaW5uZXIgZml0cyAoY29udGV4dCBmcm9tIHRoZSBvdGhlciBpbm5lciBncm91cHMsIHF1ZXJ5IHRoZSBoZWxkLW91dApncm91cCwgZm9yIHRoZSB0aHJlc2hvbGQgYW5kIHRoZSBpc290b25pYyBjYWxpYnJhdGlvbiBkb25lIGxvY2FsbHkpIGFuZCAxIGZpbmFsIGZpdCAoY29udGV4dApmcm9tIGFsbCB0cmFpbmluZyB0dXJiaW5lcywgcXVlcnkgdGhlIHRlc3QgdHVyYmluZXMpLgoKQ29udGV4dCBzaXplOiBLdW1vIFRhYnVsYXIgd2FzIHRyYWluZWQgb24gdGFibGVzIG9mIHVwIHRvIDYwLDAwMCByb3dzIGFuZCAxMDAgY29sdW1ucyBhbmQgTlZJRElBCndhcm5zIHRoYXQgYWNjdXJhY3kgbWF5IGRlZ3JhZGUgZmFyIG91dHNpZGUgdGhvc2UgcmFuZ2VzOyBubyBoYXJkIGluZmVyZW5jZSBsaW1pdCBpcwpkb2N1bWVudGVkLiBTbzogYXQgbW9zdCAxMDAgc2hvcnRsaXN0IGNvbHVtbnMsIGFuZCBgLS1jb250ZXh0LXJvd3NgIHJvd3MgcGVyIGVzdGltYXRvcgooZGVmYXVsdCAxMCwyNDAsIHRoZSBsYXJnZXN0IHNpemUgb2YgdGhlIHNlY29uZCB0cmFpbmluZyBzdGFnZSksIGAtLWVzdGltYXRvcnNgIHJhbmRvbSBjb250ZXh0CnNhbXBsZXMgYXZlcmFnZWQuCgpQZXJtdXRhdGlvbiBpbXBvcnRhbmNlIChub3QgS2VybmVsU0hBUCk6IG9uIHRoZSBmaW5hbCBtb2RlbCBvZiBlYWNoIGZvbGQsIG9uIGAtLXBlcm0tcm93c2AKbGFiZWxsZWQgdGVzdCBhbmNob3JzLCB0aGUgZHJvcCBvZiB0aGUgYW5jaG9yLWxldmVsIFJPQy1BVUMgd2hlbiBvbmUgY29sdW1uIGlzIHNodWZmbGVkLgoKT3V0cHV0IChgLS1vdXRgKTogYGt1bW9fPGZhcm0+X3Njb3Jlcy5wYXJxdWV0YCAocm93LCBzdGFnZSwgZm9sZCwgZ3JvdXAsIHApIGFuZApga3Vtb188ZmFybT5fcGVybS5jc3ZgLCBga3Vtb188ZmFybT5fdGltaW5nLmNzdmAuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50KSkKZnJvbSBjYXJlX2NhdGJvb3N0XzE1bWluIGltcG9ydCBhdWNfZmFzdCwgaW5uZXJfZ3JvdXBzLCB0YXJnZXQgICMgbm9xYTogRTQwMgoKCmRlZiB0b19wcm9iKG91dCkgLT4gbnAubmRhcnJheToKICAgICIiIgogICAgUChjbGFzcyAxKSBmcm9tIEt1bW8ncyBvdXRwdXQgdGFibGUsIHdoYXRldmVyIHRoZSBjb2x1bW4gbGF5b3V0IHR1cm5lZCBvdXQgdG8gYmUuCgogICAgOnBhcmFtIG91dDogVGhlIGBUYWJsZVRlbnNvcmAgcmV0dXJuZWQgYnkgdGhlIG1vZGVsLgoKICAgIDpyZXR1cm5zIHA6IFByb2JhYmlsaXRpZXMgb2YgdGhlIHBvc2l0aXZlIGNsYXNzLgogICAgIiIiCiAgICBpZiBoYXNhdHRyKG91dCwgInRvX3BhbmRhcyIpOiAgICAgICAgICAgICAgICAgICAgIyBUYWJsZVRlbnNvcjogb25lIHByb2JhYmlsaXR5IGNvbHVtbiBwZXIgY2xhc3MKICAgICAgICBkZiA9IG91dC50b19wYW5kYXMoKQogICAgZWxpZiBoYXNhdHRyKG91dCwgImRldGFjaCIpOiAgICAgICAgICAgICAgICAgICAgICMgYSBwbGFpbiB0b3JjaCB0ZW5zb3IgKHJvd3MsIGNsYXNzZXMpCiAgICAgICAgYSA9IG91dC5kZXRhY2goKS5mbG9hdCgpLmNwdSgpLm51bXB5KCkKICAgICAgICByZXR1cm4gYVs6LCAtMV0gaWYgYS5uZGltID09IDIgZWxzZSBhCiAgICBlbHNlOgogICAgICAgIGRmID0gcGQuRGF0YUZyYW1lKG91dCkKICAgIGNvbHMgPSBbc3RyKGMpIGZvciBjIGluIGRmLmNvbHVtbnNdCiAgICBmb3IgYyBpbiAoIjEiLCAiMS4wIiwgInByb2JfMSIsICJ0YXJnZXRfMSIsICJwMSIpOgogICAgICAgIGlmIGMgaW4gY29sczoKICAgICAgICAgICAgcmV0dXJuIGRmW2RmLmNvbHVtbnNbY29scy5pbmRleChjKV1dLnRvX251bXB5KGZsb2F0KQogICAgbnVtID0gZGYuc2VsZWN0X2R0eXBlcygibnVtYmVyIikKICAgIGlmIG51bS5zaGFwZVsxXSA9PSAyOgogICAgICAgIHJldHVybiBudW0uaWxvY1s6LCAxXS50b19udW1weShmbG9hdCkKICAgIGlmIG51bS5zaGFwZVsxXSA9PSAxOgogICAgICAgIHJldHVybiBudW0uaWxvY1s6LCAwXS50b19udW1weShmbG9hdCkKICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmtub3duIEt1bW8gb3V0cHV0IGNvbHVtbnM6IHtjb2xzfSIpCgoKY2xhc3MgS3VtbzoKICAgICIiIlRoaW4gd3JhcHBlcjogb25lIGNvbnRleHQsIG1hbnkgcXVlcmllcywgUChvbnNldCB3aXRoaW4gNyBkKS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgc2l6ZTogc3RyLCBlc3RpbWF0b3JzOiBpbnQsIHNlZWQ6IGludCk6CiAgICAgICAgaW1wb3J0IHNkbSAgIyBzdHJ1Y3R1cmVkLWRhdGEtbW9kZWxzCiAgICAgICAgaW1wb3J0IHRvcmNoCgogICAgICAgIHNlbGYuc2RtLCBzZWxmLnRvcmNoID0gc2RtLCB0b3JjaAogICAgICAgIHNlbGYubW9kZWwgPSBzZG0ubW9kZWxzLkt1bW9UYWJ1bGFyKHRhc2s9ImNsYXNzaWZpY2F0aW9uIiwgc2l6ZT1zaXplLCBkZXZpY2U9ImN1ZGEiKQogICAgICAgIHNlbGYubW9kZWwuZXZhbCgpCiAgICAgICAgc2VsZi5lc3RpbWF0b3JzID0gZXN0aW1hdG9ycwogICAgICAgIHNlbGYuYmF0Y2ggPSA0MDk2CiAgICAgICAgc2VsZi5nZW4gPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPSJjdWRhIikubWFudWFsX3NlZWQoc2VlZCkKCiAgICBkZWYgdGFibGUoc2VsZiwgWDogbnAubmRhcnJheSwgbmFtZXM6IGxpc3Rbc3RyXSwgeTogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lKToKICAgICAgICBkZiA9IHBkLkRhdGFGcmFtZShYLCBjb2x1bW5zPW5hbWVzKQogICAgICAgIG92ZXIgPSB7YzogIm51bWVyaWNhbCIgZm9yIGMgaW4gbmFtZXN9ICAgICAgICMgS3VtbyBUYWJ1bGFyIHRha2VzIG51bWVyaWNhbCBpbnB1dHMgb25seQogICAgICAgIGlmIHkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGRmWyJ0YXJnZXQiXSA9IHBkLkNhdGVnb3JpY2FsKHkuYXN0eXBlKGludCkpCiAgICAgICAgICAgIG92ZXJbInRhcmdldCJdID0gImNhdGVnb3JpY2FsIgogICAgICAgIHJldHVybiBzZWxmLnNkbS5UYWJsZVRlbnNvci5mcm9tX3BhbmRhcyhkZj1kZiwgc3R5cGVzPXNlbGYuc2RtLmluZmVyX3N0eXBlcyhkZiwgb3ZlcnJpZGVzPW92ZXIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkZXZpY2U9ImN1ZGEiKQoKICAgIGRlZiBmaXQoc2VsZiwgWDogbnAubmRhcnJheSwgeTogbnAubmRhcnJheSwgbmFtZXM6IGxpc3Rbc3RyXSkgLT4gTm9uZToKICAgICAgICB0ID0gc2VsZi50YWJsZShYLCBuYW1lcywgeSkKICAgICAgICAjIHNhbWUgYXV0b2Nhc3QgYXMgcHJlZGljdDogdGhlIGNhY2hlZCBjb250ZXh0IGtleXMvdmFsdWVzIG11c3QgbWF0Y2ggdGhlIHF1ZXJ5IGR0eXBlCiAgICAgICAgd2l0aCBzZWxmLnRvcmNoLm5vX2dyYWQoKSwgc2VsZi50b3JjaC5hdXRvY2FzdCgiY3VkYSIsIGR0eXBlPXNlbGYudG9yY2guZmxvYXQxNik6CiAgICAgICAgICAgIHNlbGYubW9kZWwuZml0KHg9dC5kcm9wX2NvbHVtbnMoInRhcmdldCIpLCB5PXRbInRhcmdldCJdLCBudW1fZXN0aW1hdG9ycz1zZWxmLmVzdGltYXRvcnMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGdlbmVyYXRvcj1zZWxmLmdlbikKCiAgICBkZWYgcHJlZGljdChzZWxmLCBYOiBucC5uZGFycmF5LCBuYW1lczogbGlzdFtzdHJdKSAtPiBucC5uZGFycmF5OgogICAgICAgICIiIlF1ZXJ5IGluIGJhdGNoZXM7IG1lbW9yeSBncm93cyB3aXRoIGJhdGNoIHggY29udGV4dCwgc28gaGFsdmUgdGhlIGJhdGNoIG9uIE9PTS4iIiIKICAgICAgICBvdXQsIGkgPSBbXSwgMAogICAgICAgIHdpdGggc2VsZi50b3JjaC5ub19ncmFkKCksIHNlbGYudG9yY2guYXV0b2Nhc3QoImN1ZGEiLCBkdHlwZT1zZWxmLnRvcmNoLmZsb2F0MTYpOgogICAgICAgICAgICB3aGlsZSBpIDwgbGVuKFgpOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIG91dC5hcHBlbmQodG9fcHJvYihzZWxmLm1vZGVsLnByZWRpY3Qoc2VsZi50YWJsZShYW2k6aSArIHNlbGYuYmF0Y2hdLCBuYW1lcykpKSkKICAgICAgICAgICAgICAgICAgICBpICs9IHNlbGYuYmF0Y2gKICAgICAgICAgICAgICAgIGV4Y2VwdCBzZWxmLnRvcmNoLk91dE9mTWVtb3J5RXJyb3I6CiAgICAgICAgICAgICAgICAgICAgc2VsZi50b3JjaC5jdWRhLmVtcHR5X2NhY2hlKCkKICAgICAgICAgICAgICAgICAgICBhc3NlcnQgc2VsZi5iYXRjaCA+IDY0LCAib3V0IG9mIG1lbW9yeSBldmVuIGF0IDY0IHF1ZXJpZXMiCiAgICAgICAgICAgICAgICAgICAgc2VsZi5iYXRjaCAvLz0gMgogICAgICAgICAgICAgICAgICAgIHByaW50KGYiICBPT006IHF1ZXJ5IGJhdGNoIC0+IHtzZWxmLmJhdGNofSIsIGZsdXNoPVRydWUpCiAgICAgICAgcmV0dXJuIG5wLmNvbmNhdGVuYXRlKG91dCkgaWYgb3V0IGVsc2UgbnAuemVyb3MoMCkKCiAgICBkZWYgY2xlYXIoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLm1vZGVsLmNsZWFyKCkKCgpkZWYgY29udGV4dF9yb3dzKHJvd3M6IG5wLm5kYXJyYXksIHk6IG5wLm5kYXJyYXksIG46IGludCwgcG9zX3NoYXJlOiBmbG9hdCwgcm5nKSAtPiBucC5uZGFycmF5OgogICAgIiIiCiAgICBTYW1wbGUgdGhlIGNvbnRleHQ6IHBvc2l0aXZlcyB1cCB0byBgcG9zX3NoYXJlYCBvZiBgbmAsIHRoZSByZXN0IG5lZ2F0aXZlcy4KCiAgICA6cGFyYW0gcm93czogQ2FuZGlkYXRlIGFuY2hvciBpbmRpY2VzIChmaW5pdGUgdGFyZ2V0KS4KICAgIDpwYXJhbSB5OiBUYXJnZXQgcGVyIGFuY2hvci4KICAgIDpwYXJhbSBuOiBDb250ZXh0IHJvd3MuCiAgICA6cGFyYW0gcG9zX3NoYXJlOiBMYXJnZXN0IHNoYXJlIG9mIHBvc2l0aXZlcy4KICAgIDpwYXJhbSBybmc6IEdlbmVyYXRvci4KCiAgICA6cmV0dXJucyByb3dzOiBUaGUgc2FtcGxlLCBzb3J0ZWQuCiAgICAiIiIKICAgIHBvcywgbmVnID0gcm93c1t5W3Jvd3NdID09IDFdLCByb3dzW3lbcm93c10gPT0gMF0KICAgIG5fcG9zID0gbWluKGxlbihwb3MpLCBpbnQobiAqIHBvc19zaGFyZSkpCiAgICBuX25lZyA9IG1pbihsZW4obmVnKSwgbiAtIG5fcG9zKQogICAgcmV0dXJuIG5wLnNvcnQobnAucl9bcm5nLmNob2ljZShwb3MsIG5fcG9zLCByZXBsYWNlPUZhbHNlKSwgcm5nLmNob2ljZShuZWcsIG5fbmVnLCByZXBsYWNlPUZhbHNlKV0pCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJmb2xkZXIgd2l0aCBhbmNob3JzXzxmYXJtPi5wYXJxdWV0LCBYXzxmYXJtPi5ucHksIGZlYXR1cmVzXzxmYXJtPi5qc29uIikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWZvbGRzIiwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0iY3ZfZm9sZHMucGFycXVldCIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1zaG9ydGxpc3QiLCByZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWZhcm1zIiwgZGVmYXVsdD0iQSxCLEMiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tb25seS1mb2xkcyIsIGRlZmF1bHQ9ImFsbCIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1zaXplIiwgZGVmYXVsdD0ibGFyZ2UiLCBjaG9pY2VzPVsic21hbGwiLCAibWVkaXVtIiwgImxhcmdlIl0pCiAgICBwLmFkZF9hcmd1bWVudCgiLS1lc3RpbWF0b3JzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWNvbnRleHQtcm93cyIsIHR5cGU9aW50LCBkZWZhdWx0PTEwXzI0MCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXBvcy1zaGFyZSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4yNSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXBlcm0tcm93cyIsIHR5cGU9aW50LCBkZWZhdWx0PTUwMDApCiAgICBwLmFkZF9hcmd1bWVudCgiLS1uby1pbm5lciIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9InNraXAgdGhlIGlubmVyIGZpdHMgKHRpbWluZyB0ZXN0cyBvbmx5KSIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCkKICAgIGFyZ3MgPSBwLnBhcnNlX2FyZ3MoKQogICAgb3V0ID0gUGF0aChhcmdzLm91dCkKICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBmb2xkcyA9IHBkLnJlYWRfcGFycXVldChhcmdzLmZvbGRzKQogICAgc2hvcnQgPSBqc29uLmxvYWRzKFBhdGgoYXJncy5zaG9ydGxpc3QpLnJlYWRfdGV4dCgpKQogICAga3VtbyA9IEt1bW8oYXJncy5zaXplLCBhcmdzLmVzdGltYXRvcnMsIGFyZ3Muc2VlZCkKICAgIGZvciBmYXJtIGluIGFyZ3MuZmFybXMuc3BsaXQoIiwiKToKICAgICAgICBBID0gcGQucmVhZF9wYXJxdWV0KFBhdGgoYXJncy5kYXRhKSAvIGYiYW5jaG9yc197ZmFybX0ucGFycXVldCIpCiAgICAgICAgWF9hbGwgPSBucC5sb2FkKFBhdGgoYXJncy5kYXRhKSAvIGYiWF97ZmFybX0ubnB5IiwgbW1hcF9tb2RlPSJyIikKICAgICAgICBuYW1lc19hbGwgPSBqc29uLmxvYWRzKChQYXRoKGFyZ3MuZGF0YSkgLyBmImZlYXR1cmVzX3tmYXJtfS5qc29uIikucmVhZF90ZXh0KCkpCiAgICAgICAgbmFtZXMgPSBzaG9ydFtmYXJtXVs6MTAwXQogICAgICAgIFggPSBucC5hc2NvbnRpZ3VvdXNhcnJheShYX2FsbFs6LCBbbmFtZXNfYWxsLmluZGV4KGMpIGZvciBjIGluIG5hbWVzXV0pCiAgICAgICAgZGVsIFhfYWxsCiAgICAgICAgeSA9IHRhcmdldChBLCA3KQogICAgICAgIHR1cmIgPSBBWyJ0dXJiaW5lIl0udG9fbnVtcHkoKQogICAgICAgIGV2bCA9IEFbImV2YWwiXS50b19udW1weSgpCiAgICAgICAgbGFiID0gbnAuZmxhdG5vbnplcm8obnAuaXNmaW5pdGUoeSkpCiAgICAgICAgZm8gPSBmb2xkc1soZm9sZHNbInNjaGVtZSJdID09ICJwZXJfZmFybSIpICYgKGZvbGRzWyJmYXJtIl0gPT0gZmFybSkgJiAoZm9sZHNbInNlZWQiXSA9PSBhcmdzLnNlZWQpXQogICAgICAgIGZvbGQgPSBwZC5TZXJpZXModHVyYikubWFwKGZvLnNldF9pbmRleCgidHVyYmluZSIpWyJmb2xkIl0pLnRvX251bXB5KCkKICAgICAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoYXJncy5zZWVkKQogICAgICAgIHNjb3JlcywgdGltaW5nLCBwZXJtID0gW10sIFtdLCBbXQogICAgICAgIHdhbnRlZCA9IG5wLnVuaXF1ZShmb2xkKSBpZiBhcmdzLm9ubHlfZm9sZHMgPT0gImFsbCIgZWxzZSBbaW50KGYpIGZvciBmIGluIGFyZ3Mub25seV9mb2xkcy5zcGxpdCgiLCIpXQogICAgICAgIGZvciBrIGluIHdhbnRlZDoKICAgICAgICAgICAgdHIgPSBmb2xkICE9IGsKICAgICAgICAgICAgc3RhZ2VzID0gW10gaWYgYXJncy5ub19pbm5lciBlbHNlIFsKICAgICAgICAgICAgICAgICgiaW5uZXIiLCBnaSwgdHIgJiB+bnAuaXNpbih0dXJiLCBnKSwgdHIgJiBucC5pc2luKHR1cmIsIGcpICYgZXZsKQogICAgICAgICAgICAgICAgZm9yIGdpLCBnIGluIGVudW1lcmF0ZShpbm5lcl9ncm91cHMobnAudW5pcXVlKHR1cmJbdHJdKSwgYXJncy5zZWVkKSldCiAgICAgICAgICAgIHN0YWdlcy5hcHBlbmQoKCJmaW5hbCIsIC0xLCB0ciwgKGZvbGQgPT0gaykgJiBldmwpKQogICAgICAgICAgICBmb3Igc3RhZ2UsIGdpLCBjdHhfbWFzaywgcV9tYXNrIGluIHN0YWdlczoKICAgICAgICAgICAgICAgIHQwID0gdGltZS50aW1lKCkKICAgICAgICAgICAgICAgIGN0eCA9IGNvbnRleHRfcm93cyhsYWJbY3R4X21hc2tbbGFiXV0sIHksIGFyZ3MuY29udGV4dF9yb3dzLCBhcmdzLnBvc19zaGFyZSwgcm5nKQogICAgICAgICAgICAgICAgcSA9IG5wLmZsYXRub256ZXJvKHFfbWFzaykKICAgICAgICAgICAgICAgIGt1bW8uZml0KFhbY3R4XSwgeVtjdHhdLCBuYW1lcykKICAgICAgICAgICAgICAgIHRfZml0ID0gdGltZS50aW1lKCkgLSB0MAogICAgICAgICAgICAgICAgcHEgPSBrdW1vLnByZWRpY3QoWFtxXSwgbmFtZXMpCiAgICAgICAgICAgICAgICB0X3ByZWQgPSB0aW1lLnRpbWUoKSAtIHQwIC0gdF9maXQKICAgICAgICAgICAgICAgIHNjb3Jlcy5hcHBlbmQocGQuRGF0YUZyYW1lKHsicm93IjogcSwgInN0YWdlIjogc3RhZ2UsICJmb2xkIjogaywgImdyb3VwIjogZ2ksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInAiOiBwcS5hc3R5cGUobnAuZmxvYXQzMil9KSkKICAgICAgICAgICAgICAgIHRpbWluZy5hcHBlbmQoZGljdChmYXJtPWZhcm0sIGZvbGQ9aW50KGspLCBzdGFnZT1zdGFnZSwgZ3JvdXA9Z2ksIGNvbnRleHQ9bGVuKGN0eCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcXVlcmllcz1sZW4ocSksIHNlY29uZHNfZml0PXRfZml0LCBzZWNvbmRzX3ByZWRpY3Q9dF9wcmVkKSkKICAgICAgICAgICAgICAgIHByaW50KGYiW3tmYXJtfSBmb2xkIHtrfSB7c3RhZ2V9IHtnaX1dIGNvbnRleHQge2xlbihjdHgpOix9LCBxdWVyaWVzIHtsZW4ocSk6LH0sICIKICAgICAgICAgICAgICAgICAgICAgIGYiZml0IHt0X2ZpdDouMGZ9cywgcHJlZGljdCB7dF9wcmVkOi4wZn1zICh7bGVuKHEpIC8gbWF4KHRfcHJlZCwgMWUtOSk6LC4wZn0gcm93cy9zKSIsCiAgICAgICAgICAgICAgICAgICAgICBmbHVzaD1UcnVlKQogICAgICAgICAgICAgICAgaWYgc3RhZ2UgPT0gImZpbmFsIiBhbmQgYXJncy5wZXJtX3Jvd3M6CiAgICAgICAgICAgICAgICAgICAgdGUgPSBucC5mbGF0bm9uemVybygoZm9sZCA9PSBrKSAmIGV2bCAmIG5wLmlzZmluaXRlKHkpKQogICAgICAgICAgICAgICAgICAgIGlmIDAgPCB5W3RlXS5zdW0oKSA8IGxlbih0ZSk6CiAgICAgICAgICAgICAgICAgICAgICAgIHRlID0gbnAuc29ydChybmcuY2hvaWNlKHRlLCBtaW4oYXJncy5wZXJtX3Jvd3MsIGxlbih0ZSkpLCByZXBsYWNlPUZhbHNlKSkKICAgICAgICAgICAgICAgICAgICAgICAgYmFzZSA9IGF1Y19mYXN0KGt1bW8ucHJlZGljdChYW3RlXSwgbmFtZXMpLCB5W3RlXSkKICAgICAgICAgICAgICAgICAgICAgICAgWHAgPSBYW3RlXS5jb3B5KCkKICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGosIGMgaW4gZW51bWVyYXRlKG5hbWVzKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNvbCA9IFhwWzosIGpdLmNvcHkoKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgWHBbOiwgal0gPSBybmcucGVybXV0YXRpb24oY29sKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgcGVybS5hcHBlbmQoZGljdChmYXJtPWZhcm0sIGZvbGQ9aW50KGspLCBmZWF0dXJlPWMsIGJhc2VfYXVjPWJhc2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGF1Y19kcm9wPWJhc2UgLSBhdWNfZmFzdChrdW1vLnByZWRpY3QoWHAsIG5hbWVzKSwgeVt0ZV0pKSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgIFhwWzosIGpdID0gY29sCiAgICAgICAgICAgICAgICAgICAgICAgIHByaW50KGYiW3tmYXJtfSBmb2xkIHtrfV0gcGVybXV0YXRpb24gaW1wb3J0YW5jZSBvbiB7bGVuKHRlKTosfSBhbmNob3JzLCAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYiYmFzZSBBVUMge2Jhc2U6LjNmfSAoe3RpbWUudGltZSgpIC0gdDA6LjBmfXMgc2luY2UgZml0KSIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICBrdW1vLmNsZWFyKCkKICAgICAgICAgICAgcGQuY29uY2F0KHNjb3JlcykudG9fcGFycXVldChvdXQgLyBmImt1bW9fe2Zhcm19X3Njb3Jlcy5wYXJxdWV0IikKICAgICAgICAgICAgcGQuRGF0YUZyYW1lKHRpbWluZykudG9fY3N2KG91dCAvIGYia3Vtb197ZmFybX1fdGltaW5nLmNzdiIsIGluZGV4PUZhbHNlKQogICAgICAgICAgICBpZiBwZXJtOgogICAgICAgICAgICAgICAgcGQuRGF0YUZyYW1lKHBlcm0pLnRvX2NzdihvdXQgLyBmImt1bW9fe2Zhcm19X3Blcm0uY3N2IiwgaW5kZXg9RmFsc2UpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='))

In [ ]:
import base64, os
os.makedirs(os.path.dirname('/kaggle/working/dataset/wind_turbine_scada_fault/cache/lstm_attention/meta.parquet'), exist_ok=True)
open('/kaggle/working/dataset/wind_turbine_scada_fault/cache/lstm_attention/meta.parquet', 'wb').write(base64.b64decode('...'))

In [ ]:
import base64, os
os.makedirs(os.path.dirname('/kaggle/working/dataset/wind_turbine_scada_fault/cache/catboost_baseline/cv_folds.parquet'), exist_ok=True)
open('/kaggle/working/dataset/wind_turbine_scada_fault/cache/catboost_baseline/cv_folds.parquet', 'wb').write(base64.b64decode('UEFSMRUEFSwVMEwVBBUAEgAAFlQIAAAAcGVyX2Zhcm0GAAAAcG9vbGVkFQAVSBUeLBWwAxUQFQYVBhw2ACgGcG9vbGVkGAhwZXJfZmFybRERAAAAJCQDAAAAsAMBATeqZgEAFQQVHhUiTBUGFQASAAAPOAEAAABBAQAAAEIBAAAAQxUAFR4VIiwVsAMVEBUGFQYcNgAoAUMYAUEREQAAAA84AwAAALADAQI8AGwBiAICFQQV+AMVygJMFUgVABIAAPwBGAMAAABBMDAFBwAxDQcAMQkOADMFBwAyBQ4AQgkjCEIwMgEVCEIwNQkHADYJBwA3BQcJOARCMQkjCT8IQjE0ARwAQwk4AEMJOABDCSMAQwkjAEMNIwQxNQUqADEFVARDMgV3BEMyBYUEQzMFVARDMw0OCVQAMwkxBDM4BTgANAkjADQJIwA0CSMANQXSBEM1CRw8NTMDAAAAQzU1AwAAAEM1NhUAFdYCFcgBLBWwAxUQFQYVBhw2ACgDQzU2GANBMDAREQAAAKsBXAMAAACwAwEGNwAQBIIwDAQBAEEgCMNAEA0PbFEUhnEcCJIkirIsDNM0RWEYx4EgSaIoy8IwTVMyGwCAjvM8EBRFkjRNFFVVlnVdGJZlmrZtHNd1nvd9IBiGojiO7iEACSEVBBUwFSxMFQYVABIAABgAAA0BPAEAAAAAAAAAAgAAAAAAAAAVABU0FTgsFbADFRAVBhUGHBgIAgAAAAAAAAAYCAAAAAAAAAAAFgAoCAIAAAAAAAAAGAgAAAAAAAAAABERAAAAGmQDAAAAsAMBAhQAFAEUAiQAJAEkAlgAWAFYAhUEFZABFVxMFRIVABIAAEgAAA0BAAENCAACDQgAAw0IAAQNCAAFDQgABg0IPAcAAAAAAAAACAAAAAAAAAAVABXqARXcASwVsAMVEBUGFQYcGAgIAAAAAAAAABgIAAAAAAAAAAAWACgICAAAAAAAAAAYCAAAAAAAAAAAEREAAAB1NAMAAACwAwEENwARIjNEGQXoECEyA0RFJjcYIBEyQwQlRhc4MCESE0RFJgc4IgAiABEiRDMREUQRMxFEMzMARAAiAEQREQAAMzMiRCIBGyAARBERIgAzABEBBEAiIiIARDMRREQRRBEzMwAAIhUEFUAVOEwVCBUAEgAAIAQDAAkBAAEJB0AAAgAAAAAAAAAAAAAAAAAAABUAFYYBFXwsFbADFRAVBhUGHBgIAwAAAAAAAAAYCAAAAAAAAAAAFgAoCAMAAAAAAAAAGAgAAAAAAAAAABERAAAAQ2ADAAAAsAMBAgsApQpQqgCl+v9VEAED/18UAQVMGAEPr/+lVVWqVVqlUFWv/6UQARMVDCBVVapVWqVQBQAVBBl8NQAYBnNjaGVtYRUMABUMJQIYBnNjaGVtZSUATBwAAAAVDCUCGARmYXJtJQBMHAAAABUMJQIYB3R1cmJpbmUlAEwcAAAAFQQlAhgEc2VlZAAVBCUCGARmb2xkABUEJQIYCG5fZXZlbnRzABawAxkcGWwmABwVDBk1AAYQGRgGc2NoZW1lFQIWsAMW5AEWvgEmVCYIHDYAKAZwb29sZWQYCHBlcl9mYXJtEREAGSwVBBUAFQIAFQAVEBUCADwW0BcZBhkmALADAAAAJgAcFQwZNQAGEBkYBGZhcm0VAhawAxaUARacASaEAibGARw2ACgBQxgBQRERABksFQQVABUCABUAFRAVAgA8FrADGQYZJgCwAwAAACYAHBUMGTUABhAZGAd0dXJiaW5lFQIWsAMWtgcW+gQmzAUm4gIcNgAoA0M1NhgDQTAwEREAGSwVBBUAFQIAFQAVEBUCADwWkAoZBhkmALADAAAAJgAcFQQZNQAGEBkYBHNlZWQVAhawAxaAAhaAAiakCCbcBxwYCAIAAAAAAAAAGAgAAAAAAAAAABYAKAgCAAAAAAAAABgIAAAAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAsAMAAAAmABwVBBk1AAYQGRgEZm9sZBUCFrADFpwEFtoDJtYKJtwJHBgICAAAAAAAAAAYCAAAAAAAAAAAFgAoCAgAAAAAAAAAGAgAAAAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCwAwAAACYAHBUEGTUABhAZGAhuX2V2ZW50cxUCFrADFuQCFtICJooOJrYNHBgIAwAAAAAAAAAYCAAAAAAAAAAAFgAoCAMAAAAAAAAAGAgAAAAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCwAwAAABauExawAyYIFoAQABksGAZwYW5kYXMY2Qd7ImluZGV4X2NvbHVtbnMiOiBbeyJraW5kIjogInJhbmdlIiwgIm5hbWUiOiBudWxsLCAic3RhcnQiOiAwLCAic3RvcCI6IDIxNiwgInN0ZXAiOiAxfV0sICJjb2x1bW5faW5kZXhlcyI6IFt7Im5hbWUiOiBudWxsLCAiZmllbGRfbmFtZSI6IG51bGwsICJwYW5kYXNfdHlwZSI6ICJ1bmljb2RlIiwgIm51bXB5X3R5cGUiOiAic3RyIiwgIm1ldGFkYXRhIjogeyJlbmNvZGluZyI6ICJVVEYtOCJ9fV0sICJjb2x1bW5zIjogW3sibmFtZSI6ICJzY2hlbWUiLCAiZmllbGRfbmFtZSI6ICJzY2hlbWUiLCAicGFuZGFzX3R5cGUiOiAib2JqZWN0IiwgIm51bXB5X3R5cGUiOiAic3RyIiwgIm1ldGFkYXRhIjogbnVsbH0sIHsibmFtZSI6ICJmYXJtIiwgImZpZWxkX25hbWUiOiAiZmFybSIsICJwYW5kYXNfdHlwZSI6ICJvYmplY3QiLCAibnVtcHlfdHlwZSI6ICJzdHIiLCAibWV0YWRhdGEiOiBudWxsfSwgeyJuYW1lIjogInR1cmJpbmUiLCAiZmllbGRfbmFtZSI6ICJ0dXJiaW5lIiwgInBhbmRhc190eXBlIjogIm9iamVjdCIsICJudW1weV90eXBlIjogInN0ciIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAic2VlZCIsICJmaWVsZF9uYW1lIjogInNlZWQiLCAicGFuZGFzX3R5cGUiOiAiaW50NjQiLCAibnVtcHlfdHlwZSI6ICJpbnQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAiZm9sZCIsICJmaWVsZF9uYW1lIjogImZvbGQiLCAicGFuZGFzX3R5cGUiOiAiaW50NjQiLCAibnVtcHlfdHlwZSI6ICJpbnQ2NCIsICJtZXRhZGF0YSI6IG51bGx9LCB7Im5hbWUiOiAibl9ldmVudHMiLCAiZmllbGRfbmFtZSI6ICJuX2V2ZW50cyIsICJwYW5kYXNfdHlwZSI6ICJpbnQ2NCIsICJudW1weV90eXBlIjogImludDY0IiwgIm1ldGFkYXRhIjogbnVsbH1dLCAiYXR0cmlidXRlcyI6IHt9LCAiY3JlYXRvciI6IHsibGlicmFyeSI6ICJweWFycm93IiwgInZlcnNpb24iOiAiMjUuMC4wIn0sICJwYW5kYXNfdmVyc2lvbiI6ICIzLjAuMyJ9ABgMQVJST1c6c2NoZW1hGOAOLy8vLy80QUZBQUFRQUFBQUFBQUtBQTRBQmdBRkFBZ0FDZ0FBQUFBQkJBQVFBQUFBQUFBS0FBd0FBQUFFQUFnQUNnQUFBQkFFQUFBRUFBQUFBUUFBQUF3QUFBQUlBQXdBQkFBSUFBZ0FBQURvQXdBQUJBQUFBTmtEQUFCN0ltbHVaR1Y0WDJOdmJIVnRibk1pT2lCYmV5SnJhVzVrSWpvZ0luSmhibWRsSWl3Z0ltNWhiV1VpT2lCdWRXeHNMQ0FpYzNSaGNuUWlPaUF3TENBaWMzUnZjQ0k2SURJeE5pd2dJbk4wWlhBaU9pQXhmVjBzSUNKamIyeDFiVzVmYVc1a1pYaGxjeUk2SUZ0N0ltNWhiV1VpT2lCdWRXeHNMQ0FpWm1sbGJHUmZibUZ0WlNJNklHNTFiR3dzSUNKd1lXNWtZWE5mZEhsd1pTSTZJQ0oxYm1samIyUmxJaXdnSW01MWJYQjVYM1I1Y0dVaU9pQWljM1J5SWl3Z0ltMWxkR0ZrWVhSaElqb2dleUpsYm1OdlpHbHVaeUk2SUNKVlZFWXRPQ0o5ZlYwc0lDSmpiMngxYlc1eklqb2dXM3NpYm1GdFpTSTZJQ0p6WTJobGJXVWlMQ0FpWm1sbGJHUmZibUZ0WlNJNklDSnpZMmhsYldVaUxDQWljR0Z1WkdGelgzUjVjR1VpT2lBaWIySnFaV04wSWl3Z0ltNTFiWEI1WDNSNWNHVWlPaUFpYzNSeUlpd2dJbTFsZEdGa1lYUmhJam9nYm5Wc2JIMHNJSHNpYm1GdFpTSTZJQ0ptWVhKdElpd2dJbVpwWld4a1gyNWhiV1VpT2lBaVptRnliU0lzSUNKd1lXNWtZWE5mZEhsd1pTSTZJQ0p2WW1wbFkzUWlMQ0FpYm5WdGNIbGZkSGx3WlNJNklDSnpkSElpTENBaWJXVjBZV1JoZEdFaU9pQnVkV3hzZlN3Z2V5SnVZVzFsSWpvZ0luUjFjbUpwYm1VaUxDQWlabWxsYkdSZmJtRnRaU0k2SUNKMGRYSmlhVzVsSWl3Z0luQmhibVJoYzE5MGVYQmxJam9nSW05aWFtVmpkQ0lzSUNKdWRXMXdlVjkwZVhCbElqb2dJbk4wY2lJc0lDSnRaWFJoWkdGMFlTSTZJRzUxYkd4OUxDQjdJbTVoYldVaU9pQWljMlZsWkNJc0lDSm1hV1ZzWkY5dVlXMWxJam9nSW5ObFpXUWlMQ0FpY0dGdVpHRnpYM1I1Y0dVaU9pQWlhVzUwTmpRaUxDQWliblZ0Y0hsZmRIbHdaU0k2SUNKcGJuUTJOQ0lzSUNKdFpYUmhaR0YwWVNJNklHNTFiR3g5TENCN0ltNWhiV1VpT2lBaVptOXNaQ0lzSUNKbWFXVnNaRjl1WVcxbElqb2dJbVp2YkdRaUxDQWljR0Z1WkdGelgzUjVjR1VpT2lBaWFXNTBOalFpTENBaWJuVnRjSGxmZEhsd1pTSTZJQ0pwYm5RMk5DSXNJQ0p0WlhSaFpHRjBZU0k2SUc1MWJHeDlMQ0I3SW01aGJXVWlPaUFpYmw5bGRtVnVkSE1pTENBaVptbGxiR1JmYm1GdFpTSTZJQ0p1WDJWMlpXNTBjeUlzSUNKd1lXNWtZWE5mZEhsd1pTSTZJQ0pwYm5RMk5DSXNJQ0p1ZFcxd2VWOTBlWEJsSWpvZ0ltbHVkRFkwSWl3Z0ltMWxkR0ZrWVhSaElqb2diblZzYkgxZExDQWlZWFIwY21saWRYUmxjeUk2SUh0OUxDQWlZM0psWVhSdmNpSTZJSHNpYkdsaWNtRnllU0k2SUNKd2VXRnljbTkzSWl3Z0luWmxjbk5wYjI0aU9pQWlNalV1TUM0d0luMHNJQ0p3WVc1a1lYTmZkbVZ5YzJsdmJpSTZJQ0l6TGpBdU15SjlBQUFBQmdBQUFIQmhibVJoY3dBQUJnQUFBQlFCQUFEWUFBQUFyQUFBQUhBQUFBQThBQUFBQkFBQUFCVC8vLzhBQUFFQ0VBQUFBQndBQUFBRUFBQUFBQUFBQUFnQUFBQnVYMlYyWlc1MGN3QUFBQUNnLy8vL0FBQUFBVUFBQUFCSS8vLy9BQUFCQWhBQUFBQVlBQUFBQkFBQUFBQUFBQUFFQUFBQVptOXNaQUFBQUFEUS8vLy9BQUFBQVVBQUFBQjQvLy8vQUFBQkFoQUFBQUFnQUFBQUJBQUFBQUFBQUFBRUFBQUFjMlZsWkFBQUFBQUlBQXdBQ0FBSEFBZ0FBQUFBQUFBQlFBQUFBTEQvLy84QUFBRVVFQUFBQUJnQUFBQUVBQUFBQUFBQUFBY0FBQUIwZFhKaWFXNWxBS0QvLy8vWS8vLy9BQUFCRkJBQUFBQVlBQUFBQkFBQUFBQUFBQUFFQUFBQVptRnliUUFBQUFESS8vLy9FQUFVQUFnQUJnQUhBQXdBQUFBUUFCQUFBQUFBQUFFVUVBQUFBQndBQUFBRUFBQUFBQUFBQUFZQUFBQnpZMmhsYldVQUFBUUFCQUFFQUFBQQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4wGWwcAAAcAAAcAAAcAAAcAAAcAAAATQ4AAFBBUjE='))

In [ ]:
import base64, os
os.makedirs(os.path.dirname('/kaggle/working/dataset/wind_turbine_scada_fault/cache/catboost_15min/shortlist.json'), exist_ok=True)
open('/kaggle/working/dataset/wind_turbine_scada_fault/cache/catboost_15min/shortlist.json', 'wb').write(base64.b64decode('ewogIkEiOiBbCiAgInNlbnNvcl8zOF9hdmdfX21pbiIsCiAgInNlbnNvcl8yNl9hdmdfX21lYW4iLAogICJzZW5zb3JfNDJfYXZnX3Npbl9fbWF4IiwKICAic2Vuc29yXzJfYXZnX2Nvc19fbWF4IiwKICAic2Vuc29yXzJfYXZnX2Nvc19fbGFzdCIsCiAgInNlbnNvcl8yMl9hdmdfX21heCIsCiAgInNlbnNvcl8xX2F2Z19zaW5fX21heCIsCiAgInNlbnNvcl8yNl9hdmdfX21pbiIsCiAgInNlbnNvcl8yX2F2Z19zaW5fX3N0ZCIsCiAgInBvd2VyXzI5X2F2Z19fbWluIiwKICAic2Vuc29yXzI2X2F2Z19fc3RkIiwKICAid2luZF9zcGVlZF80X2F2Z19fbWF4IiwKICAic2Vuc29yXzEyX2F2Z19fbWF4IiwKICAic2Vuc29yXzQyX2F2Z19zaW5fX21pbiIsCiAgInNlbnNvcl8zNV9hdmdfX21heCIsCiAgInNlbnNvcl8zMl9hdmdfX21heCIsCiAgInJlYWN0aXZlX3Bvd2VyXzI4X2F2Z19fbWF4IiwKICAic2Vuc29yXzFfYXZnX3Npbl9fbWluIiwKICAic2Vuc29yXzJfYXZnX3Npbl9fbWF4IiwKICAic2Vuc29yXzI1X2F2Z19fbGFzdCIsCiAgInNlbnNvcl8zNF9hdmdfX21pbiIsCiAgIndpbmRfc3BlZWRfM19hdmdfX21heCIsCiAgInNlbnNvcl81X2F2Z19jb3NfX21lYW4iLAogICJzZW5zb3JfOF9hdmdfX21heCIsCiAgInNlbnNvcl8xMV9hdmdfX21pbiIsCiAgIndpbmRfc3BlZWRfNF9hdmdfX21pbiIsCiAgInNlbnNvcl81X2F2Z19zaW5fX21pbiIsCiAgInNlbnNvcl81X2F2Z19jb3NfX21pbiIsCiAgInNlbnNvcl8xOF9hdmdfX2Q2aCIsCiAgInNlbnNvcl8yX2F2Z19jb3NfX21pbiIsCiAgIndpbmRfc3BlZWRfM19hdmdfX21pbiIsCiAgInNlbnNvcl8zNV9hdmdfX3N0ZCIsCiAgInNlbnNvcl8yX2F2Z19zaW5fX21lYW4iLAogICJzZW5zb3JfMzFfYXZnX19tYXgiLAogICJzZW5zb3JfMV9hdmdfY29zX19tZWFuIiwKICAic2Vuc29yXzhfYXZnX19zdGQiLAogICJzZW5zb3JfMV9hdmdfY29zX19taW4iLAogICJzZW5zb3JfMjBfYXZnX19kNmgiLAogICJzZW5zb3JfMV9hdmdfY29zX19tYXgiLAogICJzZW5zb3JfNV9hdmdfY29zX19sYXN0IiwKICAid2luZF9zcGVlZF80X2F2Z19fc3RkIiwKICAid2luZF9zcGVlZF8zX2F2Z19fc3RkIiwKICAic2Vuc29yXzEwX2F2Z19fc3RkIiwKICAic2Vuc29yXzhfYXZnX19taW4iLAogICJzZW5zb3JfNV9hdmdfY29zX19tYXgiLAogICJzZW5zb3JfNl9hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzJfYXZnX3Npbl9fbWluIiwKICAid2luZF9zcGVlZF8zX2F2Z19fZDZoIiwKICAid2luZF9zcGVlZF8zX2F2Z19fbWVhbiIsCiAgInNlbnNvcl8yX2F2Z19zaW5fX2Q2aCIsCiAgInNlbnNvcl8yX2F2Z19jb3NfX3N0ZCIsCiAgInNlbnNvcl8yX2F2Z19jb3NfX3Nsb3BlIiwKICAid2luZF9zcGVlZF8zX2F2Z19fbGFzdCIsCiAgInNlbnNvcl8yX2F2Z19jb3NfX2Q2aCIsCiAgInNlbnNvcl84X2F2Z19fbWVhbiIsCiAgInNlbnNvcl8xX2F2Z19jb3NfX2xhc3QiLAogICJzZW5zb3JfMV9hdmdfc2luX19kNmgiLAogICJzZW5zb3JfMV9hdmdfc2luX19zbG9wZSIsCiAgInNlbnNvcl8xX2F2Z19zaW5fX3N0ZCIsCiAgInNlbnNvcl8xX2F2Z19zaW5fX21lYW4iLAogICJzZW5zb3JfMV9hdmdfc2luX19sYXN0IiwKICAic2Vuc29yXzBfYXZnX19kNmgiLAogICJzZW5zb3JfMF9hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzJfYXZnX3Npbl9fc2xvcGUiLAogICJzZW5zb3JfMl9hdmdfc2luX19sYXN0IiwKICAic2Vuc29yXzFfYXZnX2Nvc19fZDZoIiwKICAic2Vuc29yXzFfYXZnX2Nvc19fc2xvcGUiLAogICJzZW5zb3JfMV9hdmdfY29zX19zdGQiLAogICJzZW5zb3JfNl9hdmdfX2Q2aCIsCiAgInNlbnNvcl82X2F2Z19fc3RkIiwKICAic2Vuc29yXzhfYXZnX19sYXN0IiwKICAic2Vuc29yXzhfYXZnX19zbG9wZSIsCiAgInNlbnNvcl84X2F2Z19fZDZoIiwKICAic2Vuc29yXzlfYXZnX19zbG9wZSIsCiAgInNlbnNvcl85X2F2Z19fZDZoIiwKICAic2Vuc29yXzdfYXZnX19kNmgiLAogICJzZW5zb3JfN19hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzEwX2F2Z19fc2xvcGUiLAogICJzZW5zb3JfMTBfYXZnX19kNmgiLAogICJzZW5zb3JfMTFfYXZnX19sYXN0IiwKICAic2Vuc29yXzExX2F2Z19fbWVhbiIsCiAgInNlbnNvcl8xMV9hdmdfX3N0ZCIsCiAgInNlbnNvcl81X2F2Z19zaW5fX21lYW4iLAogICJzZW5zb3JfNV9hdmdfc2luX19sYXN0IiwKICAid2luZF9zcGVlZF80X2F2Z19fZDZoIiwKICAid2luZF9zcGVlZF80X2F2Z19fc2xvcGUiLAogICJ3aW5kX3NwZWVkXzRfYXZnX19tZWFuIiwKICAid2luZF9zcGVlZF80X2F2Z19fbGFzdCIsCiAgIndpbmRfc3BlZWRfM19hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzVfYXZnX3Npbl9fc3RkIiwKICAic2Vuc29yXzVfYXZnX2Nvc19fZDZoIiwKICAic2Vuc29yXzVfYXZnX2Nvc19fc2xvcGUiLAogICJzZW5zb3JfNV9hdmdfY29zX19zdGQiLAogICJzZW5zb3JfNV9hdmdfc2luX19kNmgiLAogICJzZW5zb3JfNV9hdmdfc2luX19zbG9wZSIsCiAgInNlbnNvcl81X2F2Z19zaW5fX21heCIKIF0sCiAiQiI6IFsKICAic2Vuc29yXzE1X2F2Z19fbWF4IiwKICAic2Vuc29yXzEwX2F2Z19zaW5fX21pbiIsCiAgInNlbnNvcl8xX2F2Z19kaWZmX19taW4iLAogICJzZW5zb3JfNF9hdmdfc2luX19tYXgiLAogICJzZW5zb3JfOV9hdmdfX2xhc3QiLAogICJzZW5zb3JfMzNfYXZnX19tYXgiLAogICJzZW5zb3JfMF9hdmdfZGlmZl9fbWluIiwKICAic2Vuc29yXzRfYXZnX2Nvc19fbWluIiwKICAic2Vuc29yXzIxX2F2Z19jb3NfX21pbiIsCiAgInNlbnNvcl8zN19hdmdfX21heCIsCiAgInNlbnNvcl85X2F2Z19fbWF4IiwKICAic2Vuc29yXzVfYXZnX19tZWFuIiwKICAic2Vuc29yXzI5X2F2Z19fZDZoIiwKICAic2Vuc29yXzEyX2F2Z19fbWF4IiwKICAic2Vuc29yXzFfYXZnX2RpZmZfX3N0ZCIsCiAgInNlbnNvcl8yNF9hdmdfX21lYW4iLAogICJzZW5zb3JfMjFfYXZnX2Nvc19fbWVhbiIsCiAgInNlbnNvcl81X2F2Z19fc3RkIiwKICAic2Vuc29yXzIxX2F2Z19jb3NfX3N0ZCIsCiAgInNlbnNvcl8zMF9hdmdfX21pbiIsCiAgInNlbnNvcl8xX2F2Z19kaWZmX19tYXgiLAogICJzZW5zb3JfMTJfYXZnX19tZWFuIiwKICAic2Vuc29yXzRfYXZnX2Nvc19fbGFzdCIsCiAgInNlbnNvcl8xMF9hdmdfY29zX19taW4iLAogICJzZW5zb3JfM19hdmdfZGlmZl9fc3RkIiwKICAic2Vuc29yXzQyX2F2Z19fc2xvcGUiLAogICJzZW5zb3JfNF9hdmdfY29zX19zdGQiLAogICJzZW5zb3JfMzVfYXZnX19sYXN0IiwKICAic2Vuc29yXzEwX2F2Z19jb3NfX3N0ZCIsCiAgInNlbnNvcl80M19hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzEwX2F2Z19zaW5fX2xhc3QiLAogICJzZW5zb3JfMV9hdmdfZGlmZl9fbWVhbiIsCiAgInNlbnNvcl8yNV9hdmdfX2Q2aCIsCiAgInNlbnNvcl80X2F2Z19zaW5fX21lYW4iLAogICJzZW5zb3JfMjhfYXZnX19kNmgiLAogICJzZW5zb3JfNV9hdmdfX21pbiIsCiAgInNlbnNvcl8zX2F2Z19kaWZmX19taW4iLAogICJzZW5zb3JfOF9hdmdfX2Q2aCIsCiAgInNlbnNvcl8zX2F2Z19kaWZmX19tYXgiLAogICJzZW5zb3JfMTBfYXZnX3Npbl9fbWF4IiwKICAic2Vuc29yXzJfYXZnX2RpZmZfX21pbiIsCiAgInNlbnNvcl8zX2F2Z19kaWZmX19tZWFuIiwKICAic2Vuc29yXzU2X2F2Z19fZDZoIiwKICAic2Vuc29yXzEwX2F2Z19zaW5fX21lYW4iLAogICJzZW5zb3JfNF9hdmdfc2luX19taW4iLAogICJzZW5zb3JfMTBfYXZnX2Nvc19fbWF4IiwKICAicmVhY3RpdmVfcG93ZXJfMTFfYXZnX19zbG9wZSIsCiAgInJlYWN0aXZlX3Bvd2VyXzExX2F2Z19fc3RkIiwKICAic2Vuc29yXzEwX2F2Z19jb3NfX2Q2aCIsCiAgInNlbnNvcl8xMF9hdmdfY29zX19zbG9wZSIsCiAgInNlbnNvcl8xMF9hdmdfY29zX19tZWFuIiwKICAic2Vuc29yXzEwX2F2Z19jb3NfX2xhc3QiLAogICJzZW5zb3JfMTBfYXZnX3Npbl9fZDZoIiwKICAic2Vuc29yXzNfYXZnX2RpZmZfX2xhc3QiLAogICJzZW5zb3JfNF9hdmdfY29zX19kNmgiLAogICJzZW5zb3JfNF9hdmdfY29zX19zbG9wZSIsCiAgInNlbnNvcl80X2F2Z19zaW5fX2Q2aCIsCiAgInNlbnNvcl80X2F2Z19zaW5fX3Nsb3BlIiwKICAic2Vuc29yXzRfYXZnX3Npbl9fc3RkIiwKICAic2Vuc29yXzRfYXZnX3Npbl9fbGFzdCIsCiAgInNlbnNvcl8zX2F2Z19kaWZmX19kNmgiLAogICJzZW5zb3JfM19hdmdfZGlmZl9fc2xvcGUiLAogICJzZW5zb3JfMl9hdmdfZGlmZl9fZDZoIiwKICAic2Vuc29yXzEwX2F2Z19zaW5fX3Nsb3BlIiwKICAic2Vuc29yXzJfYXZnX2RpZmZfX3Nsb3BlIiwKICAic2Vuc29yXzFfYXZnX2RpZmZfX2Q2aCIsCiAgInNlbnNvcl8xX2F2Z19kaWZmX19zbG9wZSIsCiAgInNlbnNvcl8xX2F2Z19kaWZmX19sYXN0IiwKICAic2Vuc29yXzBfYXZnX2RpZmZfX2Q2aCIsCiAgInNlbnNvcl8wX2F2Z19kaWZmX19zbG9wZSIsCiAgInNlbnNvcl81X2F2Z19fbGFzdCIsCiAgInNlbnNvcl81X2F2Z19fbWF4IiwKICAic2Vuc29yXzVfYXZnX19zbG9wZSIsCiAgInNlbnNvcl81X2F2Z19fZDZoIiwKICAic2Vuc29yXzEwX2F2Z19zaW5fX3N0ZCIsCiAgInNlbnNvcl85X2F2Z19fZDZoIiwKICAic2Vuc29yXzlfYXZnX19zbG9wZSIsCiAgInNlbnNvcl85X2F2Z19fc3RkIiwKICAic2Vuc29yXzhfYXZnX19zbG9wZSIsCiAgInNlbnNvcl84X2F2Z19fc3RkIiwKICAic2Vuc29yXzdfYXZnX19kNmgiLAogICJzZW5zb3JfN19hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzdfYXZnX19tYXgiLAogICJzZW5zb3JfN19hdmdfX21pbiIsCiAgInNlbnNvcl83X2F2Z19fc3RkIiwKICAic2Vuc29yXzdfYXZnX19tZWFuIiwKICAic2Vuc29yXzZfYXZnX19kNmgiLAogICJzZW5zb3JfNl9hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzZfYXZnX19tYXgiCiBdLAogIkMiOiBbCiAgInNlbnNvcl8yMjVfYXZnX19tZWFuIiwKICAic2Vuc29yXzIyNV9hdmdfX21pbiIsCiAgInNlbnNvcl8yN19hdmdfX21pbiIsCiAgInNlbnNvcl8yMjdfYXZnX19taW4iLAogICJzZW5zb3JfMjI1X2F2Z19fbGFzdCIsCiAgInNlbnNvcl8xMDlfYXZnX19taW4iLAogICJzZW5zb3JfMjI1X2F2Z19fbWF4IiwKICAic2Vuc29yXzgxX2F2Z19fbWF4IiwKICAic2Vuc29yXzg3X2F2Z19fbWluIiwKICAic2Vuc29yXzUzX2F2Z19fbWluIiwKICAic2Vuc29yXzIzMl9hdmdfX21lYW4iLAogICJzZW5zb3JfMTY2X2F2Z19fbWluIiwKICAic2Vuc29yXzExNF9hdmdfX21heCIsCiAgInNlbnNvcl8zX2F2Z19fbWluIiwKICAic2Vuc29yXzEwOV9hdmdfX2xhc3QiLAogICJzZW5zb3JfMTEwX2F2Z19fbGFzdCIsCiAgInNlbnNvcl8yMzJfYXZnX19tYXgiLAogICJzZW5zb3JfNF9hdmdfX21heCIsCiAgInNlbnNvcl82N19hdmdfX21pbiIsCiAgInNlbnNvcl80OF9hdmdfX21pbiIsCiAgInNlbnNvcl80X2F2Z19fbWVhbiIsCiAgInNlbnNvcl8zX2F2Z19fbWVhbiIsCiAgInNlbnNvcl80X2F2Z19fbGFzdCIsCiAgInNlbnNvcl8yMzFfYXZnX19kNmgiLAogICJzZW5zb3JfMTQyX2F2Z19fbGFzdCIsCiAgInNlbnNvcl8xMjVfYXZnX2Nvc19fbWluIiwKICAic2Vuc29yXzE0Ml9hdmdfX21lYW4iLAogICJ3aW5kX3NwZWVkXzIzN19hdmdfX2Q2aCIsCiAgInNlbnNvcl8xMjVfYXZnX2Nvc19fbWVhbiIsCiAgInNlbnNvcl8zX2F2Z19fbWF4IiwKICAic2Vuc29yXzNfYXZnX19zdGQiLAogICJzZW5zb3JfNF9hdmdfX21pbiIsCiAgInNlbnNvcl8zX2F2Z19fbGFzdCIsCiAgInNlbnNvcl8wX2F2Z19fbGFzdCIsCiAgInNlbnNvcl8zX2F2Z19fZDZoIiwKICAic2Vuc29yXzNfYXZnX19zbG9wZSIsCiAgInBvd2VyXzJfYXZnX19kNmgiLAogICJwb3dlcl8yX2F2Z19fc2xvcGUiLAogICJwb3dlcl8yX2F2Z19fbWF4IiwKICAicG93ZXJfMl9hdmdfX21pbiIsCiAgInBvd2VyXzJfYXZnX19zdGQiLAogICJwb3dlcl8yX2F2Z19fbWVhbiIsCiAgInBvd2VyXzJfYXZnX19sYXN0IiwKICAic2Vuc29yXzFfYXZnX19kNmgiLAogICJzZW5zb3JfMV9hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzFfYXZnX19tYXgiLAogICJzZW5zb3JfMV9hdmdfX21pbiIsCiAgInNlbnNvcl8xX2F2Z19fc3RkIiwKICAic2Vuc29yXzFfYXZnX19tZWFuIiwKICAic2Vuc29yXzFfYXZnX19sYXN0IiwKICAic2Vuc29yXzBfYXZnX19kNmgiLAogICJzZW5zb3JfMF9hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzBfYXZnX19tYXgiLAogICJzZW5zb3JfMF9hdmdfX21pbiIsCiAgInNlbnNvcl8wX2F2Z19fc3RkIiwKICAic2Vuc29yXzRfYXZnX19zdGQiLAogICJzZW5zb3JfNF9hdmdfX3Nsb3BlIiwKICAic2Vuc29yXzRfYXZnX19kNmgiLAogICJwb3dlcl82X2F2Z19fbWluIiwKICAicG93ZXJfNV9hdmdfX2xhc3QiLAogICJwb3dlcl82X2F2Z19fc3RkIiwKICAicG93ZXJfNl9hdmdfX21lYW4iLAogICJwb3dlcl82X2F2Z19fbGFzdCIsCiAgInBvd2VyXzVfYXZnX19kNmgiLAogICJwb3dlcl81X2F2Z19fc2xvcGUiLAogICJwb3dlcl81X2F2Z19fbWF4IiwKICAicG93ZXJfNV9hdmdfX21pbiIsCiAgInBvd2VyXzVfYXZnX19zdGQiLAogICJwb3dlcl81X2F2Z19fbWVhbiIsCiAgInNlbnNvcl8wX2F2Z19fbWVhbiIKIF0KfQ=='))

## 1. Kumo API check on random data (output layout, device)

In [ ]:
import numpy as np, pandas as pd, sdm, time
rng = np.random.default_rng(0)
df = pd.DataFrame(rng.normal(size=(600, 5)), columns=list("abcde"))
df["target"] = pd.Categorical((df.a + rng.normal(size=600) > 0).astype(int))
st = sdm.infer_stypes(df, overrides={**{c: "numerical" for c in "abcde"}, "target": "categorical"})
print(st)
t = sdm.TableTensor.from_pandas(df=df, stypes=st, device="cuda")
m = sdm.models.KumoTabular(task="classification", size="small", device="cuda")
with torch.autocast("cuda", dtype=torch.float16):
    out = m(x_context=t[:400].drop_columns("target"), y_context=t[:400, "target"], x_query=t[400:].drop_columns("target"))
print(type(out), getattr(out, "shape", None))
print(out.to_pandas().head() if hasattr(out, "to_pandas") else out[:5])
sys.path.insert(0, "/kaggle/working/fault_detection")
from care_kumo_tabular import to_prob
from sklearn.metrics import roc_auc_score
p1 = to_prob(out); print("p[:5]", p1[:5], "AUC on the toy task", roc_auc_score(df.target[400:].astype(int), p1))
m.fit(x=t[:400].drop_columns("target"), y=t[:400, "target"], num_estimators=2)
print("fit/predict path", to_prob(m.predict(t[400:].drop_columns("target")))[:5]); m.clear()

## 2. 15-minute arrays and anchor features (same code as locally)

In [ ]:
def sh(cmd):
    r = subprocess.run(cmd, shell=True, cwd="/kaggle/working", capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    assert r.returncode == 0, f"failed: {cmd}"
sh(f"python fault_detection/care_fault_forecast.py prepare --input /kaggle/input --out /tmp/prepared "
   f"--workers {os.cpu_count()} | tail -3")
sh(f"python fault_detection/care_catboost_15min.py features --data /tmp/prepared --out /tmp/cb15 --workers {os.cpu_count()}")

## 3. Kumo Tabular

In [ ]:
import time
common = ("python fault_detection/care_kumo_tabular.py --data /tmp/cb15 "
          "--folds /kaggle/working/dataset/wind_turbine_scada_fault/cache/catboost_baseline/cv_folds.parquet --shortlist /kaggle/working/dataset/wind_turbine_scada_fault/cache/catboost_15min/shortlist.json "
          "--out /kaggle/working/kumo --size large --estimators 1 --perm-rows 1000")
n_gpu = torch.cuda.device_count()
plan = {0: 'C', 1: 'A,B'}
plan = plan if n_gpu >= 2 else {0: ",".join(plan.values())}
procs = {}
for gpu, farms in plan.items():
    log = open(f"/kaggle/working/kumo_gpu{gpu}.log", "w")
    procs[gpu] = subprocess.Popen(common + f" --farms {farms}", shell=True, cwd="/kaggle/working", stdout=log,
                                  stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for gpu in plan:
        lines = [l for l in open(f"/kaggle/working/kumo_gpu{gpu}.log").read().splitlines() if l.startswith("[")]
        print(f"[{(time.time() - t0) / 60:5.1f} min] gpu{gpu}: {lines[-1] if lines else ''}", flush=True)
for gpu, p in procs.items():
    print(f"gpu{gpu} exit {p.returncode}")
    if p.returncode:
        print(open(f"/kaggle/working/kumo_gpu{gpu}.log").read()[-4000:])
assert all(p.returncode == 0 for p in procs.values())

In [ ]:
!rm -rf /kaggle/working/fault_detection /kaggle/working/dataset; ls -la /kaggle/working/kumo